In [ ]:
# @title Step 1: Environment Setup & Module Synchronization

from google.colab import drive
import os
import sys
import shutil
import logging
import glob
import collections
from datetime import datetime
import requests

# Mount Google Drive (handle if already mounted)
try:
    drive.mount('/content/drive', force_remount=False)
except RuntimeError as e:
    if "already mounted" in str(e):
        print("Drive already mounted")
    else:
        raise

# Set the root project directory in Google Drive
PROJECT_PATH = "/content/drive/MyDrive/PoseAI" # @param {type:"string"}

# Setup environment paths
FAST_LANE = "/content/fast_lane"
SRC_DIR = f"{FAST_LANE}/src"
BIN_DIR = f"{FAST_LANE}/bin"

# Create required directories
os.makedirs(SRC_DIR, exist_ok=True)
os.makedirs(f"{FAST_LANE}/logs", exist_ok=True)

print("Setting up PoseAI environment...")

# Copy module files from Google Drive to local Colab environment
required_modules = [
    'config.py',
    'consensus.py',
    'docking.py',
    'preprocessor.py',
    'site_finder.py',
    'visualizer.py'
]

drive_src = f"{PROJECT_PATH}/src"
print(f"Synchronizing modules from: {drive_src}")

for module_name in required_modules:
    src_file = f"{drive_src}/{module_name}"
    dst_file = f"{SRC_DIR}/{module_name}"

    if os.path.exists(src_file):
        shutil.copy2(src_file, dst_file)
        print(f"  Copied {module_name}")
    else:
        print(f"  Warning: {module_name} not found")

# Configure Python path and environment
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
os.environ["PATH"] += f":{BIN_DIR}:/usr/local/bin"

# Install required dependencies
print("\nInstalling Python dependencies...")
os.system('pip install -q rdkit openbabel-wheel py3Dmol hdbscan -q')
print("Dependencies installed")

# Import 3rd party libraries after installation
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign, Draw
import py3Dmol

# Install 32-bit architecture and libraries for LeDock
print("\nInstalling system dependencies (32-bit libraries for LeDock)...")
os.system('sudo dpkg --add-architecture i386')
os.system('sudo apt-get update -qq')
os.system('sudo apt-get install -y -qq libc6:i386 libncurses5:i386 libstdc++6:i386 zlib1g:i386')
print("System dependencies installed")

# Build fpocket from source
print("\nVerifying fpocket installation...")
if not os.path.exists('/usr/local/bin/fpocket'):
    print("Building fpocket from source...")
    os.system('git clone -q https://github.com/Discngine/fpocket.git /tmp/fpocket')
    os.system('cd /tmp/fpocket && make -s && sudo make install -s')
    print("fpocket compiled and installed")
else:
    print("fpocket already installed")

# Download docking engine binaries
print("\nDownloading docking engine binaries...")
os.makedirs(BIN_DIR, exist_ok=True)

# Smina
if not os.path.exists(f"{BIN_DIR}/smina"):
    print("Downloading Smina...")
    os.system(f'wget -q -L https://sourceforge.net/projects/smina/files/smina.static/download -O {BIN_DIR}/smina')
    os.system(f'chmod +x {BIN_DIR}/smina')

# Gnina
if not os.path.exists(f"{BIN_DIR}/gnina"):
    print("Downloading Gnina...")
    os.system(f'wget -q -L https://github.com/gnina/gnina/releases/download/v1.1/gnina -O {BIN_DIR}/gnina')
    os.system(f'chmod +x {BIN_DIR}/gnina')

# LeDock and LePro
if not os.path.exists(f"{BIN_DIR}/ledock"):
    print("Downloading LeDock...")
    os.system(f'wget -q -L https://www.lephar.com/download/ledock_linux_x86 -O {BIN_DIR}/ledock')
    os.system(f'chmod +x {BIN_DIR}/ledock')

if not os.path.exists(f"{BIN_DIR}/lepro"):
    print("Downloading LePro...")
    os.system(f'wget -q -L https://www.lephar.com/download/lepro_linux_x86 -O {BIN_DIR}/lepro')
    os.system(f'chmod +x {BIN_DIR}/lepro')

print("Binary downloads complete")

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(name)-24s | %(levelname)-8s | %(message)s'
)
logger = logging.getLogger("poseai")

# Import and validate configuration & custom modules
from config import get_config
config = get_config()
config.validate_all()

from preprocessor import ProteinLigandPrep, fetch_rcsb_smiles, get_ligand_centroid
from docking import EnsembleManager, EngineType
from consensus import ConsensusAnalyzer
from visualizer import DockingVisualizer

logger.info("Environment initialization complete")
logger.info(f"Modules directory: {SRC_DIR}")
logger.info(f"Binaries directory: {BIN_DIR}")

# Auto-generate environment.yml for local repository completeness
env_yml = """name: poseai
channels:
  - conda-forge
  - defaults
dependencies:
  - python>=3.8
  - rdkit
  - openbabel
  - numpy
  - pandas
  - hdbscan
  - pip
  - pip:
    - py3Dmol
"""
with open(f"{PROJECT_PATH}/environment.yml", "w") as f:
    f.write(env_yml)
logger.info(f"Generated environment.yml at {PROJECT_PATH}")


In [ ]:
# @title Step 2: PoseAI Pipeline Configuration

TARGET_PDB = "1hsg" # @param {type:"string"}
LIGAND_CODE = "MK1" # @param {type:"string"}
LIGAND_SMILES = "CC(C)(C)NC(=O)[C@@H]1C[C@@H]2CCCC[C@@H]2CN1C[C@@H](O)[C@H](Cc3ccccc3)NC(=O)c4cnccn4" # @param {type:"string"}
# (Leave LIGAND_SMILES empty to dynamically fetch reference from RCSB PDB)

EXHAUSTIVENESS = 32 # @param {type:"slider", min:1, max:64}
POSES_PER_ENGINE = 50 # @param {type:"number"}
LEDOCK_POSES = 50 # @param {type:"slider", min:5, max:100}
BOX_PADDING = 5 # @param {type:"slider", min:5, max:20}
RMSD_THRESHOLD = 2.5 # @param {type:"number"}
TIMEOUT_SECONDS = 3600 # @param {type:"slider", min:600, max:7200}

USE_GNINA = True # @param {type:"boolean"}
USE_SMINA = True # @param {type:"boolean"}
USE_LEDOCK = True # @param {type:"boolean"}

print("\u2713 Parameters loaded from Step 2")


In [ ]:
# =============================================================================
# @title Step 3: PoseAI Unified Execution Pipeline (v1.0)
# =============================================================================

# Read live parameters from Cell 2 (globals)
TARGET_PDB = globals().get('TARGET_PDB', '1iep').lower()
LIGAND_CODE = globals().get('LIGAND_CODE', 'STI')
LIGAND_SMILES = globals().get('LIGAND_SMILES', None)

# Set correct result directories for the current target
RESULTS_DIR = f"/content/fast_lane/results/{TARGET_PDB.upper()}"
DRIVE_RESULTS = f"{PROJECT_PATH}/results/{TARGET_PDB.upper()}"

print("\n" + "="*70)
print("POSEAI PIPELINE v1.0 — INITIAL RELEASE")
print("="*70)

# Master Topology / Reference Fetching (Using newly migrated function)
if not LIGAND_SMILES or LIGAND_SMILES == "":
    print(f"\nFetching reference SMILES for {LIGAND_CODE}...")
    LIGAND_SMILES = fetch_rcsb_smiles(LIGAND_CODE)
    if LIGAND_SMILES:
        print(f"\u2713 Dynamically loaded SMILES: {LIGAND_SMILES}")
    else:
        print(f"\u26a0 Could not find SMILES for {LIGAND_CODE}. Consensus auto-detection may fail.")

print("\nPipeline Parameters")
print(f"  PDB ID:              {TARGET_PDB.upper()}")
print(f"  Ligand Code:         {LIGAND_CODE}")
print(f"  Engines:             {[e for e in ['GNINA', 'SMINA', 'LEDOCK'] if globals().get(f'USE_{e}', True)]}")

# STAGE 1: Preprocessing
print("\n" + "="*70)
print("STAGE 1: Structure Preparation")
print("="*70)
prep = ProteinLigandPrep(TARGET_PDB)
if not prep.fetch_structure() or not prep.prepare_receptor() or not prep.isolate_ligand(LIGAND_CODE):
    raise RuntimeError("Failed in preprocessing stage")
print(f"\u2713 Receptor: {prep.receptor_pdbqt}\n\u2713 Ligand:   {prep.ligand_mol2}")

# STAGE 2: Pocket Detection (Using newly migrated function)
print("\n" + "="*70)
print("STAGE 2: Binding Pocket Detection")
print("="*70)
center, size = get_ligand_centroid(prep.ligand_mol2, padding=BOX_PADDING)
print(f"\u2713 Search center: ({center[0]:.1f}, {center[1]:.1f}, {center[2]:.1f})")
print(f"\u2713 Search size:   ({size[0]:.1f}, {size[1]:.1f}, {size[2]:.1f})")

# STAGE 3: Ensemble Docking
print("\n" + "="*70)
print("STAGE 3: Ensemble Docking")
print("="*70)
engines_to_run = []
if USE_GNINA: engines_to_run.append(EngineType.GNINA)
if USE_SMINA: engines_to_run.append(EngineType.SMINA)
if USE_LEDOCK: engines_to_run.append(EngineType.LEDOCK)

mgr = EnsembleManager()
results = mgr.run_ensemble(
    prep.receptor_pdbqt, prep.ligand_pdbqt, center, size, RESULTS_DIR,
    exhaustiveness=EXHAUSTIVENESS, num_modes=POSES_PER_ENGINE, receptor_pdb=prep.receptor_pdb,
    ligand_mol2=prep.ligand_mol2, n_ledock_poses=LEDOCK_POSES, timeout=TIMEOUT_SECONDS, engines=engines_to_run
)
print(f"\u2713 Ensemble complete: {sum(1 for r in results if r.success)}/{len(results)} successful")

# STAGE 4: Consensus Clustering
print("\n" + "="*70)
print("STAGE 4: Consensus Clustering")
print("="*70)
analyzer = ConsensusAnalyzer(work_dir="/content/fast_lane", rmsd_threshold=RMSD_THRESHOLD, ligand_smiles=LIGAND_SMILES)
try:
    df = analyzer.analyze_ensemble(results)

    # Pose breakdown log
    if hasattr(analyzer, '_metadata'):
        engines_loaded = [meta.get('engine', 'UNKNOWN') for meta in analyzer._metadata]
        counts = collections.Counter(engines_loaded)
        print(f"\n\u2713 Poses loaded into Consensus: {len(analyzer.all_poses)} total")
        for eng, c in counts.items():
            print(f"    - {eng}: {c} poses")
        print()

    if df is not None and not df.empty:
        print(f"\u2713 Clustering complete: {len(df)} consensus cluster(s)\n")
        print(df.to_string(index=False))

        # STAGE 5: Scoring & Visualization
        print("\nSTAGE 5: Visualization")
        best_cluster = df.iloc[0]
        print(f"\u2713 Best cluster: {int(best_cluster['Cluster'])} | Confidence: {analyzer.get_confidence_score(df)}")

        viz = DockingVisualizer(prep.receptor_pdb)
        viz.create_interactive_view()
        indices = np.where(analyzer.cluster_labels == best_cluster['Cluster'])[0].tolist()
        viz.add_consensus_cluster(analyzer.all_poses, indices, label="Consensus")
        viz.show()

        # Export
        os.makedirs(RESULTS_DIR, exist_ok=True)
        df.to_csv(os.path.join(RESULTS_DIR, "cluster_summary.csv"), index=False)
        shutil.copytree(RESULTS_DIR, DRIVE_RESULTS, dirs_exist_ok=True)
        print("\u2713 Results archived.")
    else:
        print("\u2717 No consensus clusters found.")
except Exception as e:
    print(f"\u2717 Consensus analysis failed: {e}")

print("\n" + "="*70 + "\nPIPELINE COMPLETE\n" + "="*70)

In [ ]:
# =============================================================================
# @title Step 4: Validation (Native RMSD & Overlay)
# =============================================================================

print("\n" + "="*70)
print("STAGE 4: VALIDATION (Native RMSD Calculation)")
print("="*70)

from rdkit import Chem
from rdkit.Chem import AllChem, rdMolAlign
import numpy as np
from visualizer import DockingVisualizer
from consensus import calculate_native_rmsd

try:
    # Get the best pose from the top consensus cluster
    best_cluster_id = int(df.iloc[0]['Cluster'])
    best_indices = np.where(analyzer.cluster_labels == best_cluster_id)[0]

    # Take the representative (first) pose of the consensus cluster
    top_pose_mol = analyzer.all_poses[best_indices[0]]

    # Calculate In-Place RMSD using our robust function
    rmsd = calculate_native_rmsd(top_pose_mol, prep.ligand_mol2, reference_smiles=LIGAND_SMILES)

    print(f"\u2713 Reference Ligand loaded: {prep.ligand_mol2}")
    print(f"\u2713 Top Consensus Pose extracted from Cluster {best_cluster_id}")
    print(f"\nStrict In-Place Heavy-Atom RMSD: {rmsd:.3f} \u00C5")

    if rmsd <= 2.0:
        print("  -> EXCELLENT: Pose is within the universally accepted 2.0 \u00C5 threshold for a 'success'.")
    elif rmsd <= 3.0:
        print("  -> ACCEPTABLE: Pose represents the correct general binding mode.")
    else:
        print("  -> POOR: Pose deviates significantly from the native crystal structure.")

    # Visualize the overlay
    print("\nRendering Native Overlay...")
    viz_val = DockingVisualizer(prep.receptor_pdb)
    viz_val.create_interactive_view()

    # Add Reference in Green
    with open(prep.ligand_mol2, 'r') as f:
        viz_val.view.addModel(f.read(), 'mol2')
        viz_val.view.setStyle({'model': -1}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.15}})

    # Add Predicted Pose in Cyan
    pose_block = Chem.MolToMolBlock(top_pose_mol)
    viz_val.view.addModel(pose_block, 'mol')
    viz_val.view.setStyle({'model': -1}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.15}})

    viz_val.view.zoomTo()
    viz_val.show()
    print("\u25a0 Green: Native Crystal Ligand  |  \u25a0 Cyan: Top Predicted Consensus Pose")

except Exception as e:
    print(f"\n\u2717 Validation failed: {e}")
    print("Make sure Step 3 completed successfully and variables 'prep', 'df', and 'analyzer' are in memory.")

In [ ]:
# =============================================================================
# @title Step 5: Local Dataset Batch Validation
# =============================================================================

DATASET_DIR = f"{PROJECT_PATH}/dataset"
BATCH_RESULTS_DIR = f"{PROJECT_PATH}/batch_results"
os.makedirs(BATCH_RESULTS_DIR, exist_ok=True)

print("="*70)
print("STAGE 5: LOCAL DATASET BATCH VALIDATION")
print("="*70)

results_list = []

if not os.path.exists(DATASET_DIR) or not os.listdir(DATASET_DIR):
    print(f"Dataset directory not found or empty: {DATASET_DIR}")
    print("Please ensure your target folders (e.g., '1hsg') are uploaded.")
else:
    target_folders = [f for f in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, f))]
    print(f"Found {len(target_folders)} targets in {DATASET_DIR}\n")

    for target_id in target_folders:
        target_id = target_id.lower()
        target_path = os.path.join(DATASET_DIR, target_id)
        print(f"Processing Target: {target_id.upper()}")
        print("-"*30)

        # Standard PDBbind/CASF naming convention
        receptor_file = os.path.join(target_path, f"{target_id}_protein.pdb")
        ligand_file = os.path.join(target_path, f"{target_id}_ligand.mol2")

        if not os.path.exists(receptor_file) or not os.path.exists(ligand_file):
            print(f"  -> Skipping: Missing receptor_protein.pdb or ligand.mol2 in {target_path}\n")
            results_list.append({"Target": target_id.upper(), "Status": "Missing Files", "RMSD": None, "Confidence": None, "Composition": None})
            continue

        try:
            # 1. Structure Preparation (using local files)
            # We use obabel to generate the required PDBQT files locally
            receptor_pdbqt = os.path.join(target_path, f"{target_id}_protein.pdbqt")
            ligand_pdbqt = os.path.join(target_path, f"{target_id}_ligand.pdbqt")

            print("  -> Generating PDBQT files...")
            os.system(f"obabel {receptor_file} -O {receptor_pdbqt} -xr > /dev/null 2>&1")
            os.system(f"obabel {ligand_file} -O {ligand_pdbqt} -p 7.4 > /dev/null 2>&1")

            # 2. Pocket Detection
            center, size = get_ligand_centroid(ligand_file, padding=BOX_PADDING)

            # 3. Ensemble Docking
            engines_to_run = []
            if USE_GNINA: engines_to_run.append(EngineType.GNINA)
            if USE_SMINA: engines_to_run.append(EngineType.SMINA)
            if USE_LEDOCK: engines_to_run.append(EngineType.LEDOCK)

            target_results_dir = os.path.join(BATCH_RESULTS_DIR, target_id.upper())
            os.makedirs(target_results_dir, exist_ok=True)

            print("  -> Running Docking Ensemble...")
            mgr = EnsembleManager()
            docking_results = mgr.run_ensemble(
                receptor_pdbqt, ligand_pdbqt, center, size, target_results_dir,
                exhaustiveness=EXHAUSTIVENESS, num_modes=POSES_PER_ENGINE, receptor_pdb=receptor_file,
                ligand_mol2=ligand_file, n_ledock_poses=LEDOCK_POSES, timeout=TIMEOUT_SECONDS, engines=engines_to_run
            )

            # 4. Consensus Clustering
            print("  -> Analyzing Consensus...")
            native_mol = Chem.MolFromMol2File(ligand_file, sanitize=False)
            native_smiles = Chem.MolToSmiles(native_mol) if native_mol else None

            analyzer = ConsensusAnalyzer(work_dir="/content/fast_lane", rmsd_threshold=RMSD_THRESHOLD, ligand_smiles=native_smiles)
            df = analyzer.analyze_ensemble(docking_results)

            if df is not None and not df.empty:
                best_cluster_id = int(df.iloc[0]['Cluster'])
                best_indices = np.where(analyzer.cluster_labels == best_cluster_id)[0]
                top_pose_mol = Chem.RemoveHs(analyzer.all_poses[best_indices[0]])

                # Calculate RMSD
                ref_mol = Chem.RemoveHs(AllChem.AssignBondOrdersFromTemplate(native_mol, native_mol))
                rmsd = rdMolAlign.CalcRMS(top_pose_mol, ref_mol)
                print(f"  -> Native RMSD: {rmsd:.3f} \u00C5")

                # Determine Validation Status
                if rmsd <= 2.0:
                    status = "Success"
                    print("  -> Validation: SUCCESS (RMSD <= 2.0 \u00C5)")
                elif rmsd <= 3.0:
                    status = "Acceptable"
                    print("  -> Validation: ACCEPTABLE (RMSD <= 3.0 \u00C5)")
                else:
                    status = "Poor"
                    print("  -> Validation: FAILED (RMSD > 3.0 \u00C5)")

                # Get Confidence Score
                conf_score = analyzer.get_confidence_score(df)
                print(f"  -> Predicted Confidence: {conf_score}")

                # Extract Engine Composition for Top Cluster
                composition_str = "Unknown"
                if hasattr(analyzer, '_metadata'):
                    engine_counts = collections.Counter()
                    for idx in best_indices:
                        eng = analyzer._metadata[idx].get('engine', 'UNKNOWN')
                        engine_counts[eng] += 1

                    print(f"  -> Top Cluster Engine Breakdown:")
                    breakdown_list = []
                    for eng, count in engine_counts.items():
                        print(f"       - {eng}: {count} poses")
                        breakdown_list.append(f"{eng}:{count}")
                    composition_str = ", ".join(breakdown_list)

                results_list.append({"Target": target_id.upper(), "Status": status, "RMSD": rmsd, "Confidence": conf_score, "Composition": composition_str})

                # Generate 3D HTML View
                html_path = os.path.join(target_results_dir, f"{target_id.upper()}_view.html")
                view = py3Dmol.view(width=800, height=600)
                view.setBackgroundColor('white')

                # Protein
                with open(receptor_file, 'r') as f: view.addModel(f.read(), 'pdb')
                view.setStyle({'model': 0}, {'cartoon': {'color': 'lightgray'}})

                # Native Ligand
                with open(ligand_file, 'r') as f: view.addModel(f.read(), 'mol2')
                view.setStyle({'model': 1}, {'stick': {'colorscheme': 'greenCarbon', 'radius': 0.15}})

                # Predicted Pose
                view.addModel(Chem.MolToMolBlock(top_pose_mol), 'mol')
                view.setStyle({'model': 2}, {'stick': {'colorscheme': 'cyanCarbon', 'radius': 0.15}})

                view.zoomTo({'model': 1})
                with open(html_path, 'w') as f: f.write(view._make_html())

            else:
                print("  -> No consensus clusters found.")
                results_list.append({"Target": target_id.upper(), "Status": "No Clusters", "RMSD": None, "Confidence": None, "Composition": None})

        except Exception as e:
            print(f"  -> Failed: {e}")
            results_list.append({"Target": target_id.upper(), "Status": "Error", "RMSD": None, "Confidence": None, "Composition": None})

        print("\n")

    # Display and Save summary
    if results_list:
        summary_df = pd.DataFrame(results_list)
        summary_csv = os.path.join(BATCH_RESULTS_DIR, "batch_summary.csv")
        summary_df.to_csv(summary_csv, index=False)
        print(f"Batch validation complete! Summary saved to: {summary_csv}\n")
        display(summary_df)


In [15]:
%%writefile {PROJECT_PATH}/README.md
# PoseAI: A Multi-Engine Consensus Molecular Docking Pipeline

PoseAI is a modular computational framework designed to execute and harmonize ligand-binding simulations across multiple docking scoring functions. By integrating **Smina** (empirical), **Gnina** (deep learning-based), and **LeDock** (physics-based), the pipeline identifies high-confidence binding modes through unsupervised machine learning and symmetry-corrected spatial clustering.

The system has been optimized for execution within **Google Colab** to guarantee reproducibility, abstracting away complex local dependency management and system architectures by automatically provisioning a standardized Ubuntu environment.

## System Architecture

The pipeline is composed of distinct functional modules located in the `src/` directory:

*   **`preprocessor.py`**: Automates structural retrieval, solvent/ion stripping, and engine-specific formatting (PDBQT, mol2) via Open Babel.
*   **`site_finder.py`**: Identifies binding site centroids using co-crystallized ligand coordinates or dynamic pocket detection.
*   **`docking.py`**: Orchestrates multi-threaded subprocess execution for Gnina, Smina, and LeDock with dynamic configuration generation.
*   **`consensus.py`**: Analyzes generated poses, calculates symmetry-corrected Root-Mean-Square Deviation (RMSD), executes spatial clustering, and generates consensus confidence scores.
*   **`visualizer.py`**: Generates interactive 3D native-overlay visualizations of the consensus clusters using py3Dmol.
*   **`config.py`**: Manages global pipeline variables and dynamic engine tuning.

## Project Development Roadmap & Methodology

**Phase 1: Data Integration and Feature Engineering**
*   **Standardization:** Unified extraction of binding affinities and atomic Cartesian Coordinates (x, y, z) from heterogeneous engine trajectory files.
*   **Structure Preparation:** Automated generation of required topology and partial charge formats for multi-engine compatibility.

**Phase 2: Unsupervised Learning and Consensus Clustering**
*   **Distance Metrics:** Integration of robust, symmetry-aware RMSD algorithms to accurately calculate spatial divergence between poses, accounting for ligand graph symmetry.
*   **Clustering Implementation:** Identification of spatially dense binding hotspots to evaluate cross-engine convergence.
*   **Confidence Scoring:** Assignment of a Consensus Confidence Score based on the percentage of unique engines contributing to the top cluster.

**Phase 3: Scientific Validation and Visualization**
*   **Redocking Benchmarks:** Validation of the pipeline using standard PDBbind/CASF targets (e.g., Human Abl Kinase, PDB 1IEP). Success is defined as a primary consensus cluster achieving an RMSD ≤ 2.0 Å relative to the crystal structure.
*   **Visualization Deployment:** Automated generation of 3D HTML reports for immediate structural analysis of consensus results.

## Reproducibility Instructions (For Evaluation & Grading)

To ensure complete reproducibility of the pipeline and batch validation results, please follow these steps:

### 1. Cloud Storage Setup
1. Log into your Google account and open Google Drive.
2. Create a root directory named `PoseAI` in your Drive (`MyDrive/PoseAI`).
3. Upload the `src/` directory (containing the Python modules) to `MyDrive/PoseAI/src/`.
4. Upload the `dataset/` directory to `MyDrive/PoseAI/dataset/`.

### 2. Expected Dataset Structure
For the batch validation to function properly, the `dataset/` directory must follow the standard PDBbind/CASF naming conventions. Each target requires its own subfolder containing the strictly named protein and ligand files:

```text
PoseAI/
├── dataset/
│   ├── 1hsg/
│   │   ├── 1hsg_protein.pdb
│   │   └── 1hsg_ligand.mol2
│   ├── 1iep/
│   │   ├── 1iep_protein.pdb
│   │   └── 1iep_ligand.mol2
```

### 3. Execution Environment
1. Open the `PoseAI.ipynb` Notebook in Google Colab.
2. Go to **Runtime > Change runtime type** and ensure you are using a standard Python 3 compute instance.
3. Execute the Notebook sequentially from **Step 1 to Step 5**.

### 4. Automated Dependency Management
The pipeline is designed to act as an autonomous controller. **Step 1** of the notebook will automatically:
*   Mount your Google Drive to access the `PoseAI` directory.
*   Synchronize local modules.
*   Install required Python libraries (`rdkit`, `py3Dmol`, `hdbscan`, etc.).
*   Add 32-bit architecture to the Colab Ubuntu instance (required for legacy LeDock binaries).
*   Download and configure Smina, Gnina, and LeDock executables directly into the Colab environment.
*   Generate an `environment.yml` file for local Conda environment replication.

### 5. Validation Protocol
Executing **Step 5: Local Dataset Batch Validation** will automatically benchmark the pipeline against all targets in the `dataset/` folder. It evaluates structural integrity, performs ensemble docking, clusters the results, and outputs a `batch_summary.csv` alongside 3D `.html` visualization files in a newly generated `batch_results/` directory.

## References and Citations

The PoseAI framework integrates several peer-reviewed docking engines and bioinformatics libraries. Please cite the following primary literature when utilizing this pipeline for research or analysis:

**Docking Engines**
*   **Smina:** Koes, D. R., Baumgartner, M. P., & Camacho, C. J. (2013). Lessons learned from optimizing docking scoring functions. *Journal of Chemical Information and Modeling*, 53(8), 1893–1904. https://doi.org/10.1021/ci300604z
*   **LeDock:** Zhao, H., & Caflisch, A. (2013). Molecular docking by simulated annealing and minimization. *European Journal of Medicinal Chemistry*, 61, 155–172. https://doi.org/10.1016/j.ejmech.2013.01.057
*   **Gnina:** McNutt, A., Li, Y., Meli, R., Aggarwal, R., Koes, D. R. (2025). GNINA 1.3: the next increment in molecular docking with deep learning. *Journal of Cheminformatics*. https://pubmed.ncbi.nlm.nih.gov/39837943/

**Software and Libraries**
*   **Biopython:** Cock, P. J., et al. (2009). Biopython: Freely available Python tools for computational molecular biology and bioinformatics. *Bioinformatics*, 25(11), 1422–1423. https://doi.org/10.1093/bioinformatics/btp163
*   **Open Babel:** O'Boyle, N. M., et al. (2011). Open Babel: An open chemical toolbox. *Journal of Cheminformatics*, 3(1), 33. https://doi.org/10.1186/1758-2946-3-33
*   **RDKit:** RDKit: Open-source cheminformatics. https://www.rdkit.org

**Structural Data Sources**
*   **PDBbind:** Liu, Z., et al. (2017). Forging the Basis for Developing Protein-Ligand Interaction Scoring Functions. *Accounts of Chemical Research*, 50(2): 302-309.
*   **RCSB Protein Data Bank:** Berman, H. M., et al. (2000). The Protein Data Bank. *Nucleic Acids Research*, 28(1), 235–242.


Overwriting /content/drive/MyDrive/PoseAI/README.md


## Step 6: Finalize and Commit to GitHub

Before committing to GitHub, it is crucial to exclude large datasets, binary files, and temporary execution directories. We will create a `.gitignore` file to handle this, then initialize the git repository.

In [13]:
%%writefile {PROJECT_PATH}/.gitignore
# Datasets and Results (Too large for GitHub)
dataset/
results/
batch_results/
fast_lane/

# Python Cache
__pycache__/
*.pyc
*.pyo
*.pyd

# Environment and IDE
.ipynb_checkpoints/
.env
.vscode/
.idea/

# OS generated files
.DS_Store
Thumbs.db


Writing /content/drive/MyDrive/PoseAI/.gitignore


In [14]:
%%bash
# Navigate to project path
cd /content/drive/MyDrive/PoseAI

# Initialize Git if not already initialized
if [ ! -d ".git" ]; then
  git init
  echo "Initialized empty Git repository."
else
  echo "Git repository already initialized."
fi

# Set your identity (Uncomment and replace with your details if needed)
# git config --global user.email "you@example.com"
# git config --global user.name "Your Name"

# Check status of files to be committed
git status


Initialized empty Git repository in /content/drive/MyDrive/PoseAI/.git/
Initialized empty Git repository.
On branch master

No commits yet

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	.gitignore
	DEV_LOG.md
	PoseAI.ipynb
	README.md
	bin/
	data/
	environment.yml
	environment_intel.yml
	main.py
	src/

nothing added to commit but untracked files present (use "git add" to track)


hint: Using 'master' as the name for the initial branch. This default branch name
hint: is subject to change. To configure the initial branch name to use in all
hint: of your new repositories, which will suppress this warning, call:
hint: 
hint: 	git config --global init.defaultBranch <name>
hint: 
hint: Names commonly chosen instead of 'master' are 'main', 'trunk' and
hint: 'development'. The just-created branch can be renamed via this command:
hint: 
hint: 	git branch -m <name>


### Next Steps:

1. Review the output of `git status` above to ensure no large or sensitive files are being tracked.
2. In a terminal or via Colab shell commands, run:
   ```bash
   cd /content/drive/MyDrive/PoseAI
   git add .
   git commit -m "Initial commit: PoseAI pipeline and source code"
   git branch -M main
   git remote add origin https://github.com/YOUR_USERNAME/YOUR_REPOSITORY.git
   git push -u origin main
   ```
   *(Note: You will need a GitHub Personal Access Token (PAT) or SSH key configured to push directly from Colab).*